## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [1]:
! pip install faiss-cpu


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
! pip install sentence-transformers


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

c:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [2]:
MY_BUBBLE_FILE = "anti_populist.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: anti_populist
Texte: 50


,id,agent,text
0,yt_6_Hc2S02Duw_Ugytw6-BDQ2pA_Zi-TB4AaABAg,Anti-populist,Apropo de avalansa de troli ce se devarsa si a...
1,yt_bee6nXyzJ_E_UgxQ1N1kdP_MTx8B4K14AaABAg,Anti-populist,Aceasta nu este o emisiune....este o regizare ...
2,yt_bee6nXyzJ_E_Ugyjnx0utsCXEXc94q54AaABAg,Anti-populist,La pregatit bine Putin a investit bani in Guru...
3,yt_im3QoqSgfDo_UgwOL2VI3_toiLZSDqh4AaABAg,Anti-populist,"Eu am vorbit cu susținători de ai lui CG, îs d..."
4,yt_OP3PA47JM_M_UgwrxXRV2SQUdDLdbFd4AaABAg,Anti-populist,Grindeanu este o Matriosca la fel și Simion. D...


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [3]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Apropo de avalansa de troli ce se devarsa si acum aici si fac apologia nebuniei prin invective, amenintari si minciuni. De cel putin 10 ani blochez si raportez la youtube zilnic uneori zeci de troli/boti cu narative putiniste ce probabil majoritatea vin din softul de boti fara numar, ce fac atacurile astea cibernetice. *Cam cu doua zile inainte de primul tur al alegerilor, au inceput sa se deverse in rafale, repetat, prin mai toate subiectele de pe Youtube, comentarii si indemnuri venite de la d


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [4]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 2/2 [00:00

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [5]:
# TODO student:
# Bula mea are 50 texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă numărul de dimensiuni ale fiecărui embedding vectorial

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [7]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\anti_populist
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [8]:
# TODO student:
# index.faiss există: da
# index.pkl există: da
# index.ntotal este egal cu numărul de texte: 50

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [9]:
# Text nou introdus în aplicație

input_text = "CCR a decis anularea alegerilor după suspiciuni privind influențe externe."

In [10]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [11]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.372
Text: Adică UDMR a fost cu Psd tot timpul la guvernare și acum ne mirăm că votează ce mai propune AUR?? Suntem oare așa naivi?

Rezultat 2
Scor: 0.369
Text: De la 13:00 încolo, Ioana Constantin murea de frică ca nu cumva Papahagi să dea exemplu USR ca acel partid anti-șpagă, ca nu cumva audiența să afle că există și alte partide în afară de PSD și AUR... :)

Rezultat 3
Scor: 0.353
Text: Calin Georgescu a participat la turul unu ca sa fie anulat si sa nu castige alegerile, asa face el , cere bani ca sa nu i se dea!😅😅😅

Rezultat 4
Scor: 0.301
Text: o analiză detaliată, dar trebuie să fim foarte atenți la cum abordăm subiectele politice, mai ales când e vorba de partide și conflicte. Este important să discutăm într-un mod respectuos și informat, având în vedere că astfel de subiecte pot fi foarte sensibile și pot avea un impact puternic asupra opiniei publice. Cum poate opoziția din partidu AUR să încerce să erodeze democrația? Manipularea narativului Opoziția poat

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

In [ ]:
input_text = "Georgescu este manipulator"

In [17]:
scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.372
Text: Adică UDMR a fost cu Psd tot timpul la guvernare și acum ne mirăm că votează ce mai propune AUR?? Suntem oare așa naivi?

Rezultat 2
Scor: 0.369
Text: De la 13:00 încolo, Ioana Constantin murea de frică ca nu cumva Papahagi să dea exemplu USR ca acel partid anti-șpagă, ca nu cumva audiența să afle că există și alte partide în afară de PSD și AUR... :)

Rezultat 3
Scor: 0.353
Text: Calin Georgescu a participat la turul unu ca sa fie anulat si sa nu castige alegerile, asa face el , cere bani ca sa nu i se dea!😅😅😅

Rezultat 4
Scor: 0.301
Text: o analiză detaliată, dar trebuie să fim foarte atenți la cum abordăm subiectele politice, mai ales când e vorba de partide și conflicte. Este important să discutăm într-un mod respectuos și informat, având în vedere că astfel de subiecte pot fi foarte sensibile și pot avea un impact puternic asupra opiniei publice. Cum poate opoziția din partidu AUR să încerce să erodeze democrația? Manipularea narativului Opoziția poat